In [ ]:

# ============================================================
# GENERATIVE AI LABORATORY - EXPERIMENT 9
# RAG-Based Question Answering System
# ============================================================

# STEP 1: INSTALL LIBRARIES
!pip -q install sentence-transformers faiss-cpu pypdf transformers sentencepiece accelerate pandas

# STEP 2: IMPORT LIBRARIES
import re
import textwrap
import numpy as np
import pandas as pd
import torch
import faiss

from pathlib import Path
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("CUDA available:", torch.cuda.is_available())

# STEP 3: UPLOAD DOCUMENTS
# Upload PDF or TXT files about one focused topic.
uploaded = files.upload()

# STEP 4: EXTRACT TEXT
documents = []

for filename in uploaded:
    suffix = Path(filename).suffix.lower()

    if suffix == ".txt":
        text = Path(filename).read_text(
            encoding="utf-8", errors="ignore"
        )
        documents.append({
            "title": filename,
            "source": filename,
            "page": None,
            "text": text
        })

    elif suffix == ".pdf":
        reader = PdfReader(filename)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                documents.append({
                    "title": filename,
                    "source": filename,
                    "page": page_number,
                    "text": text
                })

    else:
        print("Unsupported file skipped:", filename)

# Clean formatting
for doc in documents:
    doc["text"] = re.sub(
        r"\s+", " ", doc["text"]
    ).strip()

documents = [d for d in documents if d["text"]]

if not documents:
    raise ValueError(
        "No text extracted. Upload readable PDF or TXT files."
    )

print("\nExtracted sections/pages:", len(documents))

for doc in documents[:3]:
    print("\nSource:", doc["source"], "Page:", doc["page"])
    print(doc["text"][:300])

# STEP 5: CHUNKING
# Each chunk has approximately 180 words and 40-word overlap.
chunks = []
chunk_counter = 0
chunk_size = 180
overlap = 40
step = chunk_size - overlap

for doc_index, doc in enumerate(documents, start=1):
    words = doc["text"].split()

    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]

        if not piece:
            continue

        chunk_counter += 1

        chunks.append({
            "chunk_id": f"D{doc_index}-C{chunk_counter}",
            "document_id": f"D{doc_index}",
            "title": doc["title"],
            "source": doc["source"],
            "page": doc["page"],
            "text": " ".join(piece),
            "word_count": len(piece)
        })

if not chunks:
    raise ValueError("No document chunks created.")

chunks_df = pd.DataFrame(chunks)

print("\nTotal chunks:", len(chunks))
display(chunks_df[
    ["chunk_id", "title", "page", "word_count", "text"]
].head(10))

if len(chunks) < 15:
    print("WARNING: Add more source material to reach 15-20 chunks.")

# STEP 6: GENERATE EMBEDDINGS
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [chunk["text"] for chunk in chunks]

doc_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = doc_embeddings.shape[1]

print("\nEmbedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", dimension)
print("Embedding matrix shape:", doc_embeddings.shape)

# STEP 7: CREATE FAISS VECTOR STORE
# Inner product on normalized vectors gives cosine similarity.
vector_index = faiss.IndexFlatIP(dimension)
vector_index.add(doc_embeddings)

print("\nFAISS index created successfully.")
print("Vectors stored:", vector_index.ntotal)

# STEP 8: RETRIEVE TOP-K RELEVANT CHUNKS
def retrieve(question, k=3):

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(k, len(chunks))

    scores, indices = vector_index.search(
        query_embedding, k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):
        if index < 0:
            continue

        item = dict(chunks[int(index)])
        item["similarity_score"] = float(score)
        results.append(item)

    return results

# STEP 9: LOAD GENERATIVE LANGUAGE MODEL
# This model does not require an API key.
GENERATION_MODEL = "google/flan-t5-base"

print("\nLoading generation model:", GENERATION_MODEL)

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL
)

device = "cuda" if torch.cuda.is_available() else "cpu"

generation_model = generation_model.to(device)
generation_model.eval()

print("Generation model loaded on:", device)

# STEP 10: BUILD A GROUNDED RAG PROMPT
def build_prompt(question, retrieved_chunks):

    context_parts = []

    for item in retrieved_chunks:
        page_info = (
            f", page {item['page']}"
            if item["page"] is not None else ""
        )

        context_parts.append(
            f"[Source: {item['title']}{page_info}; "
            f"Chunk: {item['chunk_id']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the question using ONLY the context below.

If the answer is not available in the context, say:
"The provided documents do not contain enough information
to answer this question."

Do not invent facts or use outside knowledge.
Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    return prompt

# STEP 11: GENERATE ANSWER AND SHOW SOURCES
def generate_answer(question, k=3, show_context=True):

    retrieved = retrieve(question, k=k)

    if not retrieved:
        print("No relevant chunks retrieved.")
        return None

    prompt = build_prompt(question, retrieved)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = generation_model.generate(
            **inputs,
            max_new_tokens=180,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    # A simple heuristic for potentially irrelevant retrieval.
    # This threshold must be validated against your corpus.
    if retrieved[0]["similarity_score"] < 0.25:
        answer = (
            "The provided documents do not contain enough "
            "information to answer this question."
        )

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("\nGENERATED ANSWER:\n", answer)

    print("\nRETRIEVED SOURCES:")

    for item in retrieved:
        print(
            f"- Document: {item['title']} | "
            f"Page: {item['page']} | "
            f"Chunk: {item['chunk_id']} | "
            f"Similarity: {item['similarity_score']:.4f}"
        )

    if show_context:
        print("\nRETRIEVED CONTEXT:")

        for rank, item in enumerate(retrieved, start=1):
            print(
                f"\n[{rank}] {item['chunk_id']} "
                f"({item['title']})"
            )
            print(
                textwrap.shorten(
                    item["text"],
                    width=700,
                    placeholder="..."
                )
            )

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "prompt": prompt
    }

# STEP 12: ASK YOUR OWN QUESTION
question = input(
    "\nEnter your question about the uploaded documents: "
)

result = generate_answer(question, k=3)

# STEP 13: RUN SIX TEST QUESTIONS
# Replace these examples with questions about your corpus.
test_questions = [
    "What is the main topic discussed in the documents?",
    "What are the important concepts explained?",
    "Which methods or tools are described?",
    "Explain an important concept using different wording.",
    "How are the concepts or steps related to each other?",
    "What is the capital of the fictional planet Zorblax?"
]

test_results = []

for q_no, test_question in enumerate(test_questions, start=1):

    print(f"\n\nTEST CASE {q_no}")

    test_result = generate_answer(
        test_question, k=3, show_context=False
    )

    if test_result is None:
        answer = "No answer generated"
        source_ids = ""
        top_score = None
    else:
        answer = test_result["answer"]
        source_ids = ", ".join([
            item["chunk_id"]
            for item in test_result["retrieved"]
        ])
        top_score = (
            test_result["retrieved"][0]["similarity_score"]
        )

    test_results.append({
        "Q No": q_no,
        "Question": test_question,
        "Top-k": 3,
        "Answer": answer,
        "Sources": source_ids,
        "Top Similarity": top_score
    })

test_results_df = pd.DataFrame(test_results)

print("\nSIX TEST CASE RESULTS")
display(test_results_df)

# STEP 14: EVALUATION TABLE
evaluation_df = test_results_df[
    ["Q No", "Question", "Top-k", "Sources"]
].copy()

evaluation_df["Relevant?"] = "Review manually"
evaluation_df["Answer Supported?"] = "Review manually"
evaluation_df["Hallucination?"] = "Review manually"
evaluation_df["Comments"] = (
    "Compare the answer against the retrieved text."
)

print("\nEVALUATION TABLE")
display(evaluation_df)

# STEP 15: COMPARE TOP-3 AND TOP-5 RETRIEVAL
comparison_question = test_questions[0]

for k_value in [3, 5]:

    print(f"\nTOP-{k_value} RETRIEVAL")
    print("Question:", comparison_question)

    for rank, item in enumerate(
        retrieve(comparison_question, k=k_value), start=1
    ):
        print(
            f"{rank}. {item['chunk_id']} | "
            f"{item['title']} | "
            f"Score: {item['similarity_score']:.4f}"
        )

        print(
            textwrap.shorten(
                item["text"], width=250, placeholder="..."
            )
        )

# STEP 16: SAVE OUTPUT FILES
chunks_df.to_csv("rag_chunks.csv", index=False)
test_results_df.to_csv("rag_test_results.csv", index=False)
evaluation_df.to_csv("rag_evaluation_table.csv", index=False)

faiss.write_index(vector_index, "rag_faiss.index")

print("\nOutput files saved successfully.")

files.download("rag_chunks.csv")
files.download("rag_test_results.csv")
files.download("rag_evaluation_table.csv")
files.download("rag_faiss.index")

print("\nEXPERIMENT 9 COMPLETED.")
print("Verify answers and complete the evaluation table manually.")


CUDA available: False


Saving D1_NLP_Introduction.pdf to D1_NLP_Introduction.pdf
Saving D2_Machine_Learning.pdf to D2_Machine_Learning.pdf
Saving D3_Deep_Learning.pdf to D3_Deep_Learning.pdf
Saving D4_Transformers_LLMs.pdf to D4_Transformers_LLMs.pdf
Saving D5_RAG_Notes.pdf to D5_RAG_Notes.pdf

Extracted sections/pages: 11

Source: D1_NLP_Introduction.pdf Page: 1
INTRODUCTION TO NATURAL LANGUAGE PROCESSING (NLP) Natural Language Processing (NLP) is a branch of artificial intelligence that enables computers to understand, process, analyze, and generate human language. TOKENIZATION Tokenization is the process of dividing text into smaller units called tokens. 

Source: D1_NLP_Introduction.pdf Page: 2
QUESTION ANSWERING A question-answering system receives a question and produces a relevant answer. A document-based system can retrieve information from documents before generating the answer.

Source: D2_Machine_Learning.pdf Page: 1
MACHINE LEARNING FUNDAMENTALS Machine Learning (ML) is a branch of artificial int

,chunk_id,title,page,word_count,text
0,D1-C1,D1_NLP_Introduction.pdf,1,180,INTRODUCTION TO NATURAL LANGUAGE PROCESSING (N...
1,D1-C2,D1_NLP_Introduction.pdf,1,123,"linguistic information. For example, the word ..."
2,D2-C3,D1_NLP_Introduction.pdf,2,25,QUESTION ANSWERING A question-answering system...
3,D3-C4,D2_Machine_Learning.pdf,1,180,MACHINE LEARNING FUNDAMENTALS Machine Learning...
4,D3-C5,D2_Machine_Learning.pdf,1,117,TRAINING AND TESTING A dataset is often divide...
5,D4-C6,D2_Machine_Learning.pdf,2,74,"EMBEDDINGS An embedding maps an object, such a..."
6,D5-C7,D3_Deep_Learning.pdf,1,180,DEEP LEARNING AND NEURAL NETWORKS Deep Learnin...
7,D5-C8,D3_Deep_Learning.pdf,1,125,adjustable parameters that help the network le...
8,D6-C9,D3_Deep_Learning.pdf,2,130,EPOCH AND BATCH An epoch is one complete pass ...
9,D7-C10,D4_Transformers_LLMs.pdf,1,180,TRANSFORMERS AND LARGE LANGUAGE MODELS A Large...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

... (33 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


EXPERIMENT 9 COMPLETED.
Verify answers and complete the evaluation table manually.
